In [ ]:
import os
import time

from config import make_paths

# --- Control variables (edit these) ---
PIPELINE_TYPE = "DIRECT"  # DIRECT | PIVOT_LLM | PIVOT_RB | PIVOT_RB_LLM
MODEL_ID = "microsoft/wavecoder-ultra-6.7b"
HF_TOKEN = os.getenv("HF_TOKEN")
PROJECT_DIR = "drive/MyDrive/Colab Notebooks/LoResLift/HumanEval/"

KOTLIN_DATA_PATH = "data/HumanEval_kotlin_v1.1.jsonl"
JAVA_DATA_PATH   = "data/HumanEval_java_v1.1.jsonl"

# --- Derived paths ---
paths = make_paths(MODEL_ID, PIPELINE_TYPE, PROJECT_DIR)

notebook_start = time.time()
print(f"Pipeline: {PIPELINE_TYPE}  |  Model: {MODEL_ID}")

# --- Mount Drive and create output dirs ---
from google.colab import drive
drive.mount("/content/drive")
os.makedirs(paths.output_dir, exist_ok=True)
os.makedirs(paths.output_pivot_dir, exist_ok=True)

## Env setup

In [ ]:
!pip install transformers evaluate accelerate huggingface_hub jsonlines
!pip install git+https://github.com/JetBrains-Research/mxeval.git
!pip install -U datasets

!rm -rf kotlin-compiler-1.9.24.zip
!wget https://github.com/JetBrains/kotlin/releases/download/v1.9.24/kotlin-compiler-1.9.24.zip
!unzip -o kotlin-compiler-1.9.24.zip
os.environ["PATH"] += ":/content/kotlinc/bin"

## Load dataset

In [ ]:
from dataset import load_kotlin_dataset, load_java_dataset

problem_list, problem_dict, kotlin_prompt_dict, kotlin_signatures = load_kotlin_dataset(KOTLIN_DATA_PATH)
java_prompt_dict = load_java_dataset(JAVA_DATA_PATH)

print(f"Loaded {len(problem_dict)} Kotlin problems")

## Load model

In [ ]:
from model import load_model

tokenizer, model = load_model(MODEL_ID, HF_TOKEN)

# Pipeline

## Direct

In [ ]:
from pipelines import run_direct

if PIPELINE_TYPE == "DIRECT":
    run_direct(kotlin_prompt_dict, model, tokenizer, paths.output_eval_file)

## Pivot

In [ ]:
from pipelines import load_failed_samples

if PIPELINE_TYPE != "DIRECT":
    results_direct_path = paths.output_group_dir + "/" + "results-direct/output_eval.jsonl_results.jsonl"
    failed_problem_dict, failed_kotlin_samples = load_failed_samples(results_direct_path, problem_dict)
    print(f"Retrying {len(failed_problem_dict)} failed problems")

In [ ]:
from pipelines import run_pivot_java_gen

if PIPELINE_TYPE != "DIRECT":
    run_pivot_java_gen(java_prompt_dict, failed_problem_dict, model, tokenizer, paths.output_java_file)

### Pivot LLM — translate Java solution back to Kotlin

In [ ]:
from pipelines import run_pivot_llm

if PIPELINE_TYPE == "PIVOT_LLM":
    run_pivot_llm(
        failed_kotlin_samples, paths.output_java_file, kotlin_signatures,
        tokenizer, model, paths.output_dir, paths.output_eval_file,
    )

### Pivot RB — export files, translate with IntelliJ, then continue

In [ ]:
from pipelines import export_for_rb_translation

if PIPELINE_TYPE in ("PIVOT_RB", "PIVOT_RB_LLM"):
    export_for_rb_translation(paths.output_java_file, paths.output_pivot_dir)
    print("Download the 'to_translate' directory, translate with IntelliJ's built-in converter,")
    print("then upload the .kt files to an adjacent 'translated/' directory.")
    input("Press Enter when finished...")

In [ ]:
from pipelines import run_pivot_rb_cleanup

if PIPELINE_TYPE in ("PIVOT_RB", "PIVOT_RB_LLM"):
    run_pivot_rb_cleanup(
        paths.output_pivot_dir, paths.output_dir, PIPELINE_TYPE,
        paths.output_file, paths.output_eval_file,
    )

### Pivot RB+LLM — LLM fixup pass

In [ ]:
from pipelines import run_pivot_rb_llm_fixup

if PIPELINE_TYPE == "PIVOT_RB_LLM":
    run_pivot_rb_llm_fixup(
        failed_kotlin_samples, kotlin_signatures, paths.output_dir,
        tokenizer, model, paths.output_eval_file,
    )

## Evaluate

In [ ]:
from pipelines import run_evaluate

run_evaluate(paths.output_eval_file, problem_dict, paths.results_file)

In [ ]:
elapsed = int(time.time() - notebook_start)
hours, remainder = divmod(elapsed, 3600)
minutes, seconds = divmod(remainder, 60)
print(f"Total runtime: {hours:02d}:{minutes:02d}:{seconds:02d}")